<a href="https://colab.research.google.com/github/jathusiya/MachineLearningToturial/blob/master/Fake_News.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn .linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [14]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [16]:
#data pre-processing

news_dataset = pd.read_csv(
    '/content/fake_news.csv',
    encoding='latin1',
    engine='python',
    on_bad_lines='skip'
)

print(news_dataset.head())


                                       uuid ord_in_thread  \
0  6a175f46bcd24d39b3e962ad0f29936721db70db             0   
1  2bdc29d12605ef9cf3f09f9875040a7113be5d5b             0   
2  c70e149fdd53de5e61c29281100b9de0ed268bc3             0   
3  7cf7c15731ac2a116dd7f629bd57ea468ed70284             0   
4  0206b54719c7e241ffe0ad4315b808290dbe6c0f             0   

                 author                      published  \
0     Barracuda Brigade  2016-10-26T21:41:00.000+03:00   
1  reasoning with facts  2016-10-29T08:47:11.259+03:00   
2     Barracuda Brigade  2016-10-31T01:41:49.479+02:00   
3                Fed Up  2016-11-01T05:22:00.000+02:00   
4                Fed Up  2016-11-01T21:56:00.000+02:00   

                                               title  \
0  Muslims BUSTED: They Stole Millions In Govât...   
1  Re: Why Did Attorney General Loretta Lynch Ple...   
2  BREAKING: Weiner Cooperating With FBI On Hilla...   
3  PIN DROP SPEECH BY FATHER OF DAUGHTER Kidnappe...   
4  F

In [17]:
news_dataset.shape

(17948, 699)

In [18]:
news_dataset.head()

,uuid,ord_in_thread,author,published,title,text,language,crawled,site_url,country,...,Unnamed: 689,Unnamed: 690,Unnamed: 691,Unnamed: 692,Unnamed: 693,Unnamed: 694,Unnamed: 695,Unnamed: 696,Unnamed: 697,Unnamed: 698
0,6a175f46bcd24d39b3e962ad0f29936721db70db,0,Barracuda Brigade,2016-10-26T21:41:00.000+03:00,Muslims BUSTED: They Stole Millions In Govât...,Print They should pay all the back all the mon...,english,2016-10-27T01:49:27.168+03:00,100percentfedup.com,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2bdc29d12605ef9cf3f09f9875040a7113be5d5b,0,reasoning with facts,2016-10-29T08:47:11.259+03:00,Re: Why Did Attorney General Loretta Lynch Ple...,Why Did Attorney General Loretta Lynch Plead T...,english,2016-10-29T08:47:11.259+03:00,100percentfedup.com,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,c70e149fdd53de5e61c29281100b9de0ed268bc3,0,Barracuda Brigade,2016-10-31T01:41:49.479+02:00,BREAKING: Weiner Cooperating With FBI On Hilla...,Red State : \nFox News Sunday reported this mo...,english,2016-10-31T01:41:49.479+02:00,100percentfedup.com,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7cf7c15731ac2a116dd7f629bd57ea468ed70284,0,Fed Up,2016-11-01T05:22:00.000+02:00,PIN DROP SPEECH BY FATHER OF DAUGHTER Kidnappe...,Email Kayla Mueller was a prisoner and torture...,english,2016-11-01T15:46:26.304+02:00,100percentfedup.com,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0206b54719c7e241ffe0ad4315b808290dbe6c0f,0,Fed Up,2016-11-01T21:56:00.000+02:00,FANTASTIC! TRUMP'S 7 POINT PLAN To Reform Heal...,Email HEALTHCARE REFORM TO MAKE AMERICA GREAT ...,english,2016-11-01T23:59:42.266+02:00,100percentfedup.com,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
news_dataset.isnull().sum()

,0
uuid,3
ord_in_thread,1901
author,5313
published,3431
title,4502
...,...
Unnamed: 694,17947
Unnamed: 695,17947
Unnamed: 696,17947
Unnamed: 697,17947


In [20]:
#replacing null value with empty strings
news_dataset=news_dataset.fillna('')

In [21]:
from types import new_class
#merging author name and news title
news_dataset['content'] = news_dataset['author']+' '+news_dataset['title']

In [22]:
print(news_dataset[('content')])

0        Barracuda Brigade Muslims BUSTED: They Stole M...
1        reasoning with facts Re: Why Did Attorney Gene...
2        Barracuda Brigade BREAKING: Weiner Cooperating...
3        Fed Up PIN DROP SPEECH BY FATHER OF DAUGHTER K...
4        Fed Up FANTASTIC! TRUMP'S 7 POINT PLAN To Refo...
                               ...                        
17943                                           replaceme 
17944                                            Freedumb 
17945                                  major major maj... 
17946                                          beemasters 
17947                                   i&#039;m-confused 
Name: content, Length: 17948, dtype: object


In [23]:
#seperating the data& label
x= news_dataset.drop(columns='spam_score', axis=1)
y= news_dataset['spam_score']

In [24]:
print(x)
print(y)

                                           uuid ord_in_thread  \
0      6a175f46bcd24d39b3e962ad0f29936721db70db             0   
1      2bdc29d12605ef9cf3f09f9875040a7113be5d5b             0   
2      c70e149fdd53de5e61c29281100b9de0ed268bc3             0   
3      7cf7c15731ac2a116dd7f629bd57ea468ed70284             0   
4      0206b54719c7e241ffe0ad4315b808290dbe6c0f             0   
...                                         ...           ...   
17943  f1b5d0e44803f48732bde854a9fdf95837219b12             2   
17944  36011ceba3647e1bea78299b68b6fb705a1fc1ad             3   
17945  6995d1aa9ac99926106489b14b5530e85358059a             4   
17946  7de8ae90eee164eb756db6c8a3772288e11d7a94             5   
17947  dabef7095b7d9dae6eb0d83c4cbb40b85efd7ae5             6   

                     author                      published  \
0         Barracuda Brigade  2016-10-26T21:41:00.000+03:00   
1      reasoning with facts  2016-10-29T08:47:11.259+03:00   
2         Barracuda Brigade  2016

In [25]:
#stemming
port_stem = PorterStemmer()

In [26]:
def stemming(content):
  stemmed_content = re.sub('[^a-zA-Z]', ' ' ,content)
  stemmed_content=stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content=[port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content =' '.join(stemmed_content)
  return stemmed_content

In [27]:
news_dataset['content'] = news_dataset['content'].apply(stemming)

print(new

In [28]:
print(news_dataset['content'])

0        barracuda brigad muslim bust stole million gov...
1        reason fact attorney gener loretta lynch plead...
2        barracuda brigad break weiner cooper fbi hilla...
3        fed pin drop speech father daughter kidnap kil...
4        fed fantast trump point plan reform healthcar ...
                               ...                        
17943                                             replacem
17944                                             freedumb
17945                                      major major maj
17946                                              beemast
17947                                               confus
Name: content, Length: 17948, dtype: object


In [29]:
#seperating the data and label
x= news_dataset['content'].values
y= news_dataset['spam_score'].values

In [30]:
print(x)

['barracuda brigad muslim bust stole million gov benefit'
 'reason fact attorney gener loretta lynch plead fifth'
 'barracuda brigad break weiner cooper fbi hillari email investig' ...
 'major major maj' 'beemast' 'confus']


In [31]:
print(y)

['0' '0' '0' ... '0' '0' '0']


In [32]:
#converting the textual data to numerical data
vectorizer =TfidfVectorizer()
vectorizer.fit(x)

x= vectorizer.transform(x)

In [33]:
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 128986 stored elements and shape (17948, 14946)>
  Coords	Values
  (0, 1086)	0.4174400964201443
  (0, 1226)	0.34129977279767576
  (0, 1660)	0.3986410931600137
  (0, 1818)	0.34601883040311354
  (0, 5442)	0.35513911569741885
  (0, 8414)	0.2710687810526029
  (0, 8773)	0.26851011549687853
  (0, 12685)	0.3986410931600137
  (1, 839)	0.3610106010148262
  (1, 4524)	0.3201649374949169
  (1, 4715)	0.40677694326636055
  (1, 5221)	0.3083901261198526
  (1, 7750)	0.38136132498902314
  (1, 7848)	0.35258592700163804
  (1, 10081)	0.38735629162968566
  (1, 10773)	0.29461095449822866
  (2, 1086)	0.4665183965190104
  (2, 1615)	0.27083444222768094
  (2, 1660)	0.4455092003917537
  (2, 2766)	0.39689270919865577
  (2, 4081)	0.2495622249612789
  (2, 4617)	0.2458912481212036
  (2, 5973)	0.18823850164002368
  (2, 6634)	0.2832943217852179
  (2, 14465)	0.33848752425776174
  :	:
  (17931, 12468)	1.0
  (17932, 7007)	0.7071067811865475
  (17932, 12395)	0.7

In [34]:
#spliting the dataset to train and test
x_train,x_test,y_train, y_test = train_test_split(x,y,test_size = 0.2, stratify=y, random_state=2)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(x_train, y_train)

In [ ]:
#Evaluation
#accuracy score on the training data
x_train_prediction = model.predict(x_train)
trainning_data_accuracy = accuracy_score(x_train_prediction, y_train)

In [ ]:
print('Accuracy score of the trainning data: ', trainning_data_accuracy)